<a href="https://colab.research.google.com/github/akashkumarcs24-cloud/aiLAB/blob/main/FirstOrderResolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re

# =====================================================
# FIRST ORDER LOGIC RESOLUTION
# Natural Language -> FOL -> CNF -> Resolution -> Proof Tree
# =====================================================

# ------------------ UNIFICATION ------------------

def unify(x, y, subs=None):
    if subs is None:
        subs = {}

    if x == y:
        return subs

    if isinstance(x, str) and x.startswith('?'):
        subs[x] = y
        return subs

    if isinstance(y, str) and y.startswith('?'):
        subs[y] = x
        return subs

    return None


# ------------------ CLAUSE REPRESENTATION ------------------

def literal(neg, predicate, *args):
    return (neg, predicate, tuple(args))


def display_literal(lit):
    neg, pred, args = lit
    return ("~" if neg else "") + pred + "(" + ", ".join(args) + ")"


def display_clause(clause):
    if not clause:
        return "{}"
    return " OR ".join(display_literal(x) for x in clause)


# ------------------ NATURAL LANGUAGE TO FOL ------------------

def convert_to_fol(sentence):

    s = sentence.lower().strip().rstrip(".")

    # John likes all kind of food
    if "likes all kind" in s:
        return (
            "FOR ALL x: Food(x) -> Likes(John,x)",
            [[literal(True, "Food", "?x"),
              literal(False, "Likes", "john", "?x")]]
        )

    # Apple and vegetables are food
    if "and" in s and "are food" in s:
        return (
            "Food(Apple) AND Food(Vegetables)",
            [
                [literal(False, "Food", "apple")],
                [literal(False, "Food", "vegetables")]
            ]
        )

    # Anything anyone eats and not killed is food
    if "eats" in s and "not killed" in s and "food" in s:
        return (
            "FOR ALL x,y: (Eats(y,x) AND NotKilled(y)) -> Food(x)",
            [[
                literal(True, "Eats", "?y", "?x"),
                literal(True, "NotKilled", "?y"),
                literal(False, "Food", "?x")
            ]]
        )

    # Anil eats peanuts and still alive
    if "eats" in s and "alive" in s:
        match = re.search(r"(\w+) eats (\w+)", s)

        if match:
            person, item = match.groups()

            return (
                f"Eats({person},{item}) AND Alive({person})",
                [
                    [literal(False, "Eats", person, item)],
                    [literal(False, "Alive", person)]
                ]
            )

    # Harry eats everything that Anil eats
    if "everything that" in s:
        match = re.search(
            r"(\w+) eats everything that (\w+) eats", s
        )

        if match:
            person1, person2 = match.groups()

            return (
                f"FOR ALL x: Eats({person2},x) -> Eats({person1},x)",
                [[
                    literal(True, "Eats", person2, "?x"),
                    literal(False, "Eats", person1, "?x")
                ]]
            )

    # Anyone who is alive implies not killed
    if "alive implies not killed" in s:
        return (
            "FOR ALL x: Alive(x) -> NotKilled(x)",
            [[
                literal(True, "Alive", "?x"),
                literal(False, "NotKilled", "?x")
            ]]
        )

    # Anyone who is not killed implies alive
    if "not killed implies alive" in s:
        return (
            "FOR ALL x: NotKilled(x) -> Alive(x)",
            [[
                literal(True, "NotKilled", "?x"),
                literal(False, "Alive", "?x")
            ]]
        )

    # General atomic facts
    match = re.fullmatch(r"(\w+) is alive", s)
    if match:
        return (
            f"Alive({match.group(1)})",
            [[literal(False, "Alive", match.group(1))]]
        )

    match = re.fullmatch(r"(\w+) is food", s)
    if match:
        return (
            f"Food({match.group(1)})",
            [[literal(False, "Food", match.group(1))]]
        )

    match = re.fullmatch(r"(\w+) eats (\w+)", s)
    if match:
        a, b = match.groups()
        return (
            f"Eats({a},{b})",
            [[literal(False, "Eats", a, b)]]
        )

    raise ValueError("Sentence pattern not supported: " + sentence)


# ------------------ QUERY CONVERSION ------------------

def convert_query(query):

    match = re.fullmatch(
        r"\s*(\w+) likes (\w+)\.?\s*",
        query.lower()
    )

    if match:
        a, b = match.groups()
        return literal(False, "Likes", a, b)

    raise ValueError("Query format should be: John likes peanuts")


# ------------------ RESOLUTION ------------------

def apply_substitution(lit, subs):
    neg, pred, args = lit

    new_args = []

    for arg in args:
        while arg in subs:
            arg = subs[arg]
        new_args.append(arg)

    return (neg, pred, tuple(new_args))


def resolve(c1, c2):

    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):

            neg1, pred1, args1 = l1
            neg2, pred2, args2 = l2

            if pred1 != pred2 or neg1 == neg2:
                continue

            if len(args1) != len(args2):
                continue

            subs = {}

            for a, b in zip(args1, args2):
                result = unify(a, b, subs)

                if result is None:
                    break
            else:
                new_clause = []

                for lit in c1[:i] + c1[i+1:] + c2[:j] + c2[j+1:]:
                    new_clause.append(
                        apply_substitution(lit, subs)
                    )

                # Remove duplicate literals
                new_clause = list(dict.fromkeys(new_clause))

                return new_clause

    return None


# ------------------ PROOF TREE ------------------

def resolution_proof(clauses):

    proof_tree = []

    for i, clause in enumerate(clauses):
        proof_tree.append({
            "clause": clause,
            "parent": None
        })

    known = [tuple(map(tuple, c)) for c in clauses]

    step = 1

    while step <= 500:

        new_found = False

        for i in range(len(proof_tree)):

            for j in range(i + 1, len(proof_tree)):

                c1 = proof_tree[i]["clause"]
                c2 = proof_tree[j]["clause"]

                result = resolve(c1, c2)

                if result is None:
                    continue

                key = tuple(result)

                if key in known:
                    continue

                known.append(key)

                proof_tree.append({
                    "clause": result,
                    "parent": (i, j)
                })

                new_index = len(proof_tree) - 1

                print(
                    f"Step {step}: "
                    f"C{i+1} + C{j+1} -> "
                    f"C{new_index+1}: "
                    f"{display_clause(result)}"
                )

                step += 1
                new_found = True

                if len(result) == 0:
                    print("\nEMPTY CLAUSE {} DERIVED!")
                    return proof_tree, new_index, True

        if not new_found:
            break

    return proof_tree, None, False


# ------------------ DISPLAY PROOF TREE ------------------

def print_tree(tree, index, level=0, visited=None):

    if visited is None:
        visited = set()

    if index in visited:
        return

    visited.add(index)

    node = tree[index]

    print(
        "    " * level +
        f"C{index+1}: {display_clause(node['clause'])}"
    )

    if node["parent"] is not None:

        p1, p2 = node["parent"]

        print("    " * (level + 1) + "Derived from:")

        print_tree(tree, p1, level + 2, visited)
        print_tree(tree, p2, level + 2, visited)


# ------------------ MAIN PROGRAM ------------------

print("=" * 60)
print("       FIRST ORDER LOGIC RESOLUTION PROVER")
print("=" * 60)

n = int(input("\nEnter number of knowledge base sentences: "))

knowledge_base = []

for i in range(n):
    sentence = input(f"Enter sentence {i+1}: ")
    knowledge_base.append(sentence)


query = input("\nEnter query to prove: ")


# STEP 1: FOL CONVERSION

print("\n========== STEP 1: FOL REPRESENTATION ==========")

clauses = []

for i, sentence in enumerate(knowledge_base):

    fol, cnf = convert_to_fol(sentence)

    print(f"\nSentence: {sentence}")
    print(f"FOL: {fol}")

    clauses.extend(cnf)


# STEP 2: CNF

print("\n========== STEP 2: CNF CONVERSION ==========")

for i, clause in enumerate(clauses):
    print(f"C{i+1}: {display_clause(clause)}")


# STEP 3: NEGATE QUERY

print("\n========== STEP 3: NEGATE QUERY ==========")

query_literal = convert_query(query)

negated_query = (
    not query_literal[0],
    query_literal[1],
    query_literal[2]
)

print("Original Query:", display_literal(query_literal))
print("Negated Query:", display_literal(negated_query))

clauses.append([negated_query])


# STEP 4: RESOLUTION

print("\n========== STEP 4: RESOLUTION ==========")

tree, final_index, proved = resolution_proof(clauses)


# STEP 5: PROOF TREE

print("\n========== STEP 5: PROOF TREE ==========")

if proved:

    print_tree(tree, final_index)

    print("\n====================================")
    print("QUERY SUCCESSFULLY PROVED!")
    print("The empty clause {} is derived.")
    print("====================================")

else:

    print("Could not derive the empty clause.")
    print("Query could not be proved.")

       FIRST ORDER LOGIC RESOLUTION PROVER

Enter number of knowledge base sentences: 7
Enter sentence 1: John likes all kind of food
Enter sentence 2: Apple and vegetables are food
Enter sentence 3: Anything anyone eats and not killed is food
Enter sentence 4: Anil eats peanuts and still alive
Enter sentence 5: Harry eats everything that Anil eats
Enter sentence 6: Anyone who is alive implies not killed
Enter sentence 7: Anyone who is not killed implies alive

Enter query to prove: John likes peanuts

========== STEP 1: FOL REPRESENTATION ==========

Sentence: John likes all kind of food
FOL: FOR ALL x: Food(x) -> Likes(John,x)

Sentence: Apple and vegetables are food
FOL: Food(Apple) AND Food(Vegetables)

Sentence: Anything anyone eats and not killed is food
FOL: FOR ALL x,y: (Eats(y,x) AND NotKilled(y)) -> Food(x)

Sentence: Anil eats peanuts and still alive
FOL: Eats(anil,peanuts) AND Alive(anil)

Sentence: Harry eats everything that Anil eats
FOL: FOR ALL x: Eats(anil,x) -> Eats(h